# Reconnaissance d'entités nommées dans TEI/XML avec l'API OpenAI

Ce notebook effectue une **reconnaissance d'entités nommées (NER)** semi-automatique sur des textes français du XVIe siècle encodés en TEI/XML (le corpus François Rabelais) à l'aide d'un grand modèle de langue servi par **[OpenAI](https://platform.openai.com/)**.

**Pipeline.** Chaque paragraphe (`<p>`) d'un fichier TEI *brut* est envoyé au modèle avec une invite à quelques exemples (*few-shot prompt*). Le modèle encapsule les noms de personnes, de lieux, de zones géographiques et d'organisations dans les balises `<persName>`, `<placeName>`, `<geogName>` et `<orgName>`, sans modifier les autres balises. Les paragraphes annotés sont réinsérés dans l'arbre TEI puis sauvegardés.

## Prérequis (en locale)

1. Installer les dépendances (une seule fois) :

   ```
   pip install openai lxml
   ```

2. Définir votre clé API OpenAI dans l'environnement **avant** de lancer Jupyter :

   ```
   export OPENAI_API_KEY="your_key_here"
   ```

In [ ]:
# À exécuter une seule fois si les paquets ne sont pas déjà installés :
# %pip install -q openai lxml

In [ ]:
import os
from lxml import etree
from openai import OpenAI


# === Configuration =========================================================

# --- gestion des tokens / fenêtre de contexte ------------------------------
# max_tokens limite uniquement la SORTIE ; il doit tenir à côté du prompt
# dans la fenêtre de contexte du modèle :  prompt_tokens + max_tokens <= CONTEXT_WINDOW.
MODEL_CONTEXT = {
    "gpt-4o": 128000,
    "gpt-4o-mini": 128000,
    "gpt-4.1": 1047576,
    "gpt-4.1-mini": 1047576,
}

MODEL = "gpt-4o"
#MODEL = "gpt-4o-mini"

CONTEXT_WINDOW = MODEL_CONTEXT.get(MODEL, 128000)   # valeur de repli conservative pour les modèles inconnus


INPUT_PATH = "plain/B751131011_RES_P_Y2_1349_tei_plain.xml"
OUTPUT_PATH = "output/B751131011_RES_P_Y2_1349_tei_annotated_openai.xml"
MAX_PARAGRAPHS = 5        # nombre de <p> à annoter par exécution (mettre None pour tout annoter)
MAX_TOKENS = 8000          # nombre maximum de tokens de sortie par paragraphe
TEMPERATURE = 0.0          # sortie quasi-déterministe
USE_EXAMPLES = True
TEI_NS = "http://www.tei-c.org/ns/1.0"
NS = {"tei": TEI_NS}

In [ ]:
# === Client OpenAI =========================================================
#api_key = os.environ.get("OPENAI_API_KEY")
api_key = ""
if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY n'est pas définie. Exécutez  export OPENAI_API_KEY='votre_clé'  "
        "puis redémarrez le kernel."
    )

client = OpenAI(api_key=api_key)
print("Client OpenAI prêt. Modèle :", MODEL)

In [ ]:
# === Chargement du document TEI brut et sélection des paragraphes ==========
tree = etree.parse(INPUT_PATH)
root = tree.getroot()

paragraphs = root.xpath("//tei:text//tei:p", namespaces=NS)

if MAX_PARAGRAPHS is not None:
    paragraphs = paragraphs[:MAX_PARAGRAPHS]

print(f"Loaded {INPUT_PATH}")
print(f"Annotating {len(paragraphs)} <p> element(s).")

In [ ]:
# === Prompt few-shot et fonction d'annotation ===============================
SYSTEM_PROMPT = """You are an expert in TEI/XML encoding and named entity recognition for 16th-century French texts."""

# Les définitions des éléments sont les gloses courtes en anglais tirées des directives TEI P5
# (https://www.tei-c.org/release/doc/tei-p5-doc/en/html/). Les ensembles de valeurs @type sont
# ceux effectivement utilisés dans ce corpus.
INSTRUCTIONS = """You will receive a single <p> element from a TEI/XML document written in 16th-century French. Wrap the named entities with the TEI elements below (definitions from the TEI P5 guidelines). Add the @type attribute, using one of the listed values, when the category is clear; use no other attributes (never invent key / ref / xml:id).


Main entity elements:
  - <persName>: a proper noun or proper-noun phrase referring to a person, possibly including the person's forenames, surnames, honorifics, added names, etc.
  - <placeName>: an absolute or relative place name. type one of: ville, pays, batiment, region, domaine.
  - <geogName>: a name associated with some geographical feature, such as a valley, mountain or river. type one of: geo, hydro.
  - <orgName>: an organizational name (an organisation or a people). type one of: peuple, communaute.

Nesting:
  - Split a person's name into the components above only when they are clearly identifiable; never force forename/surname onto single-token names or epithets.
  - Nest entities when natural: a place inside a person's title (le comte de <placeName>...</placeName>), or a person inside a place name.

Coverage:
  - Include role titles (roy ..., comte de ...).
  - Annotate religious persons and roles (frere Jean, pape) like the others; use <roleName type='religieux'> for religious titles.

Rules:
  - Output must be well-formed XML; never introduce unescaped < or > inside attribute values; never leave unmatched opening or closing tags.
  - Keep EVERY existing tag exactly as it is (<choice>, <orig>, <reg>, <lb/>, <pb/>, <hi>, <seg>, <ptr/>, ...). Do not remove, change, reorder or add anything else.
  - Output ONLY the <p> element, nothing before or after it.
  - Names may be split in more than one line.
  - Do NOT wrap the answer in <root>, <document> or markdown code fences.
  - Do NOT invent Namespace prefixes."""

EXAMPLES = """Input: <p rend="indent">Or poursuiz ce propos torcheculatif
               <lb/>je <choice><orig>ten</orig><reg>t’en</reg></choice> prie: Et par ma barbe pour un
               <lb/>bussart tu auras soixante pippes, <choice><orig>Jen-
               <lb rend="hyphen"/>tends</orig><reg>J’en-
               <lb rend="hyphen"/>tends</reg></choice> de ce bon vin Breton, lequel poinct
               <lb/>ne croist en Bretaigne, Mais en ce bon
               <lb/>pays de Verron.
        </p>
Output: <p rend="indent">Or poursuiz ce propos torcheculatif
               <lb/>je <choice><orig>ten</orig><reg>t’en</reg></choice> prie: Et par ma barbe pour un
               <lb/>bussart tu auras soixante pippes, <choice><orig>Jen-
               <lb rend="hyphen"/>tends</orig><reg>J’en-
               <lb rend="hyphen"/>tends</reg></choice> de ce bon vin Breton, lequel poinct
               <lb/>ne croist en <placeName>Bretaigne</placeName>, Mais en ce bon
               <lb/>pays de <placeName>Verron</placeName>.
        </p>

Input: <p>
               <lb/><hi rend="larger">C</hi>Es propos entenduz le bon
               <lb/>homme Grandgousier fut
               <lb/>ravy en admiration conside
               <lb rend="hyphen"/>rant le hault sens &amp; mer-
               <lb rend="hyphen"/>veilleux entendement de
               <lb/>son filz Gargantua.
        </p>

Output: <p>
               <lb/><hi rend="larger">C</hi>Es propos entenduz le bon
               <lb/>homme <persName>Grandgousier</persName> fut
               <lb/>ravy en admiration conside
               <lb rend="hyphen"/>rant le hault sens &amp; mer-
               <lb rend="hyphen"/>veilleux entendement de
               <lb/>son filz <persName>Gargantua</persName>.
        </p>

Input: <p rend="indent">Et dist a ses gouvernantes: Philip-
               <lb rend="hyphen"/>pe roy de Macedone congneut le bon
               <lb/>sens de son filz Alexandre, a manier dex-


          <pb facs="http://www.bvh.univ-tours.fr/B360446201_B343_2/ecran/B360446201_B343_2_0080.jpg" xml:id="B360446201_B343_2_0080" n="[40v]"/>
               <lb rend="hyphen"/>trement un cheval. Car ledict cheval
               <lb/>estoit si terrible &amp; efrené que nul ausoit
               <lb/>monter dessus: Par ce que a tous ses
               <lb/>chevaucheurs il bailloit la saccade: a <choice><orig>lun</orig><reg>l’un</reg></choice>
               <lb/>rompant le coul, a <choice><orig>laultre</orig><reg>l’aultre</reg></choice> les jambes,
               <lb/>a <choice><orig>laultre</orig><reg>l’aultre</reg></choice> la cervelle, a <choice><orig>laultre</orig><reg>l’aultre</reg></choice> les man-
               <lb rend="hyphen"/>dibules. Ce que considerant Alexandre
               <lb/>en <choice><orig>lhippodrame</orig><reg>l’hippodrame</reg></choice> (qui estoit le lieu ou <choice><orig>lon</orig><reg>l’on</reg></choice>
               <lb/>pourmenoit &amp; voultigeoit les chevaulx)
               <lb/>advisa que la fureur du cheval ne ve-
               <lb rend="hyphen"/>noit que de frayeur <choice><orig>quil</orig><reg>qu’il</reg></choice> prenoit a son
               <lb/>umbre. Dont montant dessus le feist
               <lb/>courir encontre le Soleil, si que <choice><orig>lumbre</orig><reg>l’umbre</reg></choice>
               <lb/>tumboit par derriere, &amp; par ce moyen rendit
               <lb/>le cheval doulx a son vouloir. A quoy
               <lb/>congneut son pere le divin entendement
               <lb/>qui en luy estoit &amp; le feist tresbien endo-
               <lb rend="hyphen"/>ctriner par <persName>Aristoteles</persName> qui pour lors estoit
               <lb/>estimé sus tous philosophes de Grece.
        </p>

Output: <p rend="indent">Et dist a ses gouvernantes: <persName>Philip-
               <lb rend="hyphen"/>pe</persName> roy de <placeName type="pays">Macedone</placeName> congneut le bon
               <lb/>sens de son filz <persName>Alexandre</persName>, a manier dex-


          <pb facs="http://www.bvh.univ-tours.fr/B360446201_B343_2/ecran/B360446201_B343_2_0080.jpg" xml:id="B360446201_B343_2_0080" n="[40v]"/>
               <lb rend="hyphen"/>trement un cheval. Car ledict cheval
               <lb/>estoit si terrible &amp; efrené que nul ausoit
               <lb/>monter dessus: Par ce que a tous ses
               <lb/>chevaucheurs il bailloit la saccade: a <choice><orig>lun</orig><reg>l’un</reg></choice>
               <lb/>rompant le coul, a <choice><orig>laultre</orig><reg>l’aultre</reg></choice> les jambes,
               <lb/>a <choice><orig>laultre</orig><reg>l’aultre</reg></choice> la cervelle, a <choice><orig>laultre</orig><reg>l’aultre</reg></choice> les man-
               <lb rend="hyphen"/>dibules. Ce que considerant <persName>Alexandre</persName>
               <lb/>en <choice><orig>lhippodrame</orig><reg>l’hippodrame</reg></choice> (qui estoit le lieu ou <choice><orig>lon</orig><reg>l’on</reg></choice>
               <lb/>pourmenoit &amp; voultigeoit les chevaulx)
               <lb/>advisa que la fureur du cheval ne ve-
               <lb rend="hyphen"/>noit que de frayeur <choice><orig>quil</orig><reg>qu’il</reg></choice> prenoit a son
               <lb/>umbre. Dont montant dessus le feist
               <lb/>courir encontre le Soleil, si que <choice><orig>lumbre</orig><reg>l’umbre</reg></choice>
               <lb/>tumboit par derriere, &amp; par ce moyen rendit
               <lb/>le cheval doulx a son vouloir. A quoy
               <lb/>congneut son pere le divin entendement
               <lb/>qui en luy estoit &amp; le feist tresbien endo-
               <lb rend="hyphen"/>ctriner par <persName>Aristoteles</persName> qui pour lors estoit
               <lb/>estimé sus tous philosophes de <placeName type="pays">Grece</placeName>.
        </p>

Input: <p>
               <lb/><hi rend="larger">A</hi> Tant son pere aperceut
               <lb/>que vrayement il estudioit
               <lb/>tresbien &amp; y mettoit tout
               <lb/>son temps, toutesfoys <choice><orig>quen</orig><reg>qu’en</reg></choice>
               <lb/>rien ne prouffitoit. Et que
               <lb/>pis est, en devenoit fou, niays, tout re-
               <lb rend="hyphen"/>veux &amp; rassoté.
        </p>

Output: <p>
               <lb/><hi rend="larger">A</hi> Tant son pere aperceut
               <lb/>que vrayement il estudioit
               <lb/>tresbien &amp; y mettoit tout
               <lb/>son temps, toutesfoys <choice><orig>quen</orig><reg>qu’en</reg></choice>
               <lb/>rien ne prouffitoit. Et que
               <lb/>pis est, en devenoit fou, niays, tout re-
               <lb rend="hyphen"/>veux &amp; rassoté.
        </p>

Input: <p rend="indent">Ce pendent vint un commandeur jam
               <lb rend="hyphen"/>bonnier de sainct Antoine pour faire sa
               <lb/>queste suille: lequel pour se faire enten-
               <lb rend="hyphen"/>dre de loing, &amp; faire trembler le lard, au
               <lb/>charnier, les voulut emporter furtive-


          <pb facs="http://www.bvh.univ-tours.fr/B360446201_B343_2/ecran/B360446201_B343_2_0094.jpg" xml:id="B360446201_B343_2_0094" n="[47v]"/>
               <lb rend="hyphen"/>ment. Mais par honnesteté les laissa,
               <lb/>non par ce <choice><orig>quelles</orig><reg>qu’elles</reg></choice> estoient trop chaul-
               <lb rend="hyphen"/>des, mais par ce <choice><orig>quelles</orig><reg>qu’elles</reg></choice> estoient quel-
               <lb rend="hyphen"/>que peu trop pesantes a la portee. Cil
               <lb/>ne fut pas celluy de Bourg. Car il est
               <lb/>trop de mes amys.
        </p>

Output: <p rend="indent">Ce pendent vint un commandeur jam
               <lb rend="hyphen"/>bonnier de <orgName type="communaute">sainct Antoine</orgName> pour faire sa
               <lb/>queste suille: lequel pour se faire enten-
               <lb rend="hyphen"/>dre de loing, &amp; faire trembler le lard, au
               <lb/>charnier, les voulut emporter furtive-


          <pb facs="http://www.bvh.univ-tours.fr/B360446201_B343_2/ecran/B360446201_B343_2_0094.jpg" xml:id="B360446201_B343_2_0094" n="[47v]"/>
               <lb rend="hyphen"/>ment. Mais par honnesteté les laissa,
               <lb/>non par ce <choice><orig>quelles</orig><reg>qu’elles</reg></choice> estoient trop chaul-
               <lb rend="hyphen"/>des, mais par ce <choice><orig>quelles</orig><reg>qu’elles</reg></choice> estoient quel-
               <lb rend="hyphen"/>que peu trop pesantes a la portee. Cil
               <lb/>ne fut pas celluy de <placeName type="ville">Bourg</placeName>. Car il est
               <lb/>trop de mes amys.
        </p>"""


def build_prompt(paragraph_xml):
    if USE_EXAMPLES:
        return (
            INSTRUCTIONS
            + "\nExamples:\n\n"
            + EXAMPLES
            + "Now annotate the following paragraph. Output ONLY the <p> element.\n\n"
            + f"Input: {paragraph_xml}\n"
            + "Output: "
        )
    else:
        return (
            INSTRUCTIONS
            + "Now annotate the following paragraph. Output ONLY the <p> element.\n\n"
            + f"Input: {paragraph_xml}\n"
            + "Output: "
        )


def clean_output(text):
    """Supprime les délimiteurs Markdown / texte parasite et ne conserve que l'élément <p>...</p>."""
    text = text.strip()
    text = text.replace("&rsquo;", "\u2019")
    if text.startswith("```"):
        lines = text.splitlines()
        lines = lines[1:]                       # supprimer le délimiteur ouvrant (``` ou ```xml)
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]                  # supprimer le délimiteur fermant
        text = "\n".join(lines).strip()
    start = text.find("<p")
    end = text.rfind("</p>")
    if start != -1 and end != -1:
        text = text[start:end + len("</p>")]
    return text


def ensure_namespace(p_xml):
    """S'assurer que le <p> retourné porte l'espace de noms TEI par défaut avant l'analyse."""
    head = p_xml.split(">", 1)[0]
    if "xmlns" not in head:
        p_xml = p_xml.replace("<p", f'<p xmlns="{TEI_NS}"', 1)
    return p_xml


def approx_tokens(text):
    return len(text) // 4 + 1          # ~4 caractères/token : un garde-fou simple et indépendant du modèle


def annotate_paragraph(paragraph_xml):
    prompt = build_prompt(paragraph_xml)

    # Cette tâche renvoie le paragraphe avec quelques balises ajoutées, donc la sortie est
    # ~de la taille de l'entrée. Dimensionner max_tokens à partir de l'entrée et limiter au budget de contexte.
    prompt_tokens = approx_tokens(SYSTEM_PROMPT) + approx_tokens(prompt)
    needed = int(approx_tokens(paragraph_xml) * 1.6) + 256   # écho + quelques balises d'entités
    budget = CONTEXT_WINDOW - prompt_tokens - 128            # laisser une marge de sécurité
    if needed > budget:                                      # l'entrée elle-même est trop longue
        raise RuntimeError(
            f"paragraphe trop long pour {MODEL} (~{prompt_tokens} tokens de prompt, "
            f"contexte {CONTEXT_WINDOW}) ; le découper ou utiliser un modèle à contexte plus long"
        )
    max_out = max(256, min(needed, budget, MAX_TOKENS))

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        max_tokens=max_out,
        temperature=TEMPERATURE,
    )

    choice = response.choices[0]
    usage = response.usage
    ptok = getattr(usage, "prompt_tokens", "?")
    ctok = getattr(usage, "completion_tokens", "?")
    print(f"  finish_reason={choice.finish_reason} "
          f"prompt={ptok} completion={ctok} max_out={max_out}")
    if choice.finish_reason == "length":                     # tronqué -> ne pas l'analyser
        raise RuntimeError(
            f"sortie tronquée à {ctok} tokens (max_out={max_out}) ; "
            f"augmenter MAX_TOKENS ou utiliser un modèle à contexte plus long"
        )
    return clean_output(choice.message.content)

In [ ]:
# === Annotation de chaque paragraphe =======================================
XML_ID = "{http://www.w3.org/XML/1998/namespace}id"
NER_ERROR = "ner-error"          # marqueur utilisé sur @ana et sur <note type="...">
failures = []


def flag_failure(p, reason, raw=None):
    """Conserve le <p> original dans la sortie mais le marque comme un échec de REN.

    Le paragraphe est balisé avec @ana="#ner-error" (les retrouver avec
    //tei:p[@ana='#ner-error']) et une <note type="ner-error"> est insérée comme son
    premier enfant, contenant le message d'erreur et -- lorsque disponible -- la sortie
    brute du modèle qui n'a pas pu être analysée. Un expert peut ensuite relire le
    paragraphe, le corriger manuellement, puis supprimer la note et le drapeau @ana.
    """
    p.set("ana", f"#{NER_ERROR}")
    note = etree.Element(f"{{{TEI_NS}}}note")
    note.set("type", NER_ERROR)
    note.set("resp", "#openai-ner")
    body = f"REN ÉCHOUÉ : {reason}"
    if raw:
        body += "\n--- sortie brute du modèle (n'a pas pu être analysée) ---\n" + raw
    note.text = body
    note.tail = p.text          # préserver le texte initial original du paragraphe
    p.text = None
    p.insert(0, note)


for p in paragraphs:
    xml_id = p.get(XML_ID, "(pas de xml:id)")
    print(f"Annotation du paragraphe {xml_id} ...")

    paragraph_xml = etree.tostring(p, encoding="unicode")
    raw = None
    try:
        raw = ensure_namespace(annotate_paragraph(paragraph_xml))   # peut lever une exception (modèle/longueur)
        new_p = etree.fromstring(raw.encode("utf-8"))               # peut lever XMLSyntaxError
        p.getparent().replace(p, new_p)
    except etree.XMLSyntaxError as exc:
        reason = f"XML invalide produit par le modèle : {exc}"
        print(f"  -> {reason}\n     (original conservé, signalé dans le document avec une <note>)")
        flag_failure(p, reason, raw=raw)
        failures.append((xml_id, reason))
    except Exception as exc:
        reason = f"échec de l'appel au modèle : {exc}"
        print(f"  -> {reason}\n     (original conservé, signalé dans le document avec une <note>)")
        flag_failure(p, reason, raw=None)
        failures.append((xml_id, reason))

print("\nTerminé.")

In [ ]:
# === Sauvegarde du document annoté et compte rendu =========================
os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)
tree.write(OUTPUT_PATH, encoding="utf-8", xml_declaration=True)
print(f"Document TEI annoté sauvegardé -> {OUTPUT_PATH}")

added = tree.xpath(
    "//tei:text//tei:persName | //tei:text//tei:placeName "
    "| //tei:text//tei:geogName | //tei:text//tei:orgName",
    namespaces=NS,
)
print(f"Balises d'entités présentes dans le corps du document : {len(added)}")

flagged = tree.xpath("//tei:p[@ana='#ner-error']", namespaces=NS)
print(f"Paragraphes signalés comme erreurs de REN dans le document : {len(flagged)} "
      f"(les retrouver avec  //tei:p[@ana='#ner-error']  ou rechercher  type=\"ner-error\")")

if failures:
    print(f"\n{len(failures)} paragraphe(s) n'ont pas pu être annotés (conservés + signalés) :")
    for fid, reason in failures:
        print(f"  - {fid}: {reason}")
else:
    print("Tous les paragraphes ont été annotés sans erreur.")